# 02 — Single-step tool use

**Goal.** Connect Llama 3.1 8B (running in Ollama) to the tool registry from nb 01. Let the model answer questions by choosing **one** tool and calling it. No planning loop yet — exactly one tool call per question, then a natural-language answer.

**Why single-step first.** The think→act→observe loop in nb 03 is just this same exchange repeated until the model stops asking for tools. If you can't get one round-trip working cleanly, the loop won't either.

**Prereqs.**
- Ollama running locally (`ollama serve`)
- `llama3.1:8b` pulled (`ollama list` to confirm)
- `pip install ollama`
- nb 01 has been run at least once (to confirm tools work)

## What this notebook will and won't do

**Will:** show the raw `messages` exchange so you can see exactly what the model receives and emits. Demystify tool-use.

**Won't:** chain multiple tool calls. If the model decides one tool isn't enough, this notebook just stops and answers with what it has. nb 03 fixes that.

## 1. Setup

We re-import the tools from nb 01 by running it as a script. This is a deliberately simple way to share state between notebooks without yet refactoring into a proper module — we'll do that in nb 04.

If you'd rather not re-execute nb 01, you can copy its cells 1–6 into this notebook. The important thing is that `TOOL_SCHEMAS`, `TOOL_FUNCTIONS`, and `dispatch_tool_call` are defined.

In [1]:
# Lightweight import-by-execution pattern. Runs nb 01's code cells in this kernel.
import json, nbformat
from pathlib import Path

NB1 = Path("01_tools_not_agents.ipynb")
nb = nbformat.read(NB1, as_version=4)
for cell in nb.cells:
    if cell.cell_type == "code":
        exec(cell.source, globals())

print(f"loaded {len(TOOL_SCHEMAS)} tools:", [t['function']['name'] for t in TOOL_SCHEMAS])

connected to: /home/saqib/cti/cti_401/db/cti.duckdb
tables: ['alerts', 'documents']
title: Share and accept documents securely
truncated: False | total chars: 790
preview: Latest News
SecureDrop Inbox 1.0.0 releasedShare documents securely with these organizations
The Washington Post
The Washington Post is an American daily newspaper published in Washington, DC.
The Guardian
The Guardian is a British daily newspaper.
Disclose
A French non-profit investigative media or ...
4 tools registered:
  - query_documents
  - get_document
  - query_alerts
  - count_recent_victims
-> query_documents({'source': 'onion', 'limit': 3})
   {"count": 3, "results": [{"url_id": "f77e59ad053e076d99eecfabd4f2ed043c5c8807", "source": "onion", "label": "Hacking", "score": 0.964, "title": "Share and accept documents securely"}, {"url_id": "44f1

-> query_documents({'label': 'Hacking'})
   {"count": 1, "results": [{"url_id": "f77e59ad053e076d99eecfabd4f2ed043c5c8807", "source": "onion", "label": "Hacking", "sco

In [2]:
import ollama

MODEL = "llama3.1:8b"

# Verify the model is available locally before we try to chat with it.
available = [m["model"] for m in ollama.list().get("models", [])]
assert any(MODEL in m for m in available), (
    f"{MODEL} not in ollama list. Run: ollama pull {MODEL}"
)
print("ollama OK, using model:", MODEL)

ollama OK, using model: llama3.1:8b


## 2. The anatomy of one tool-use exchange

Three messages flow through the model on a successful single-step call:

| # | Role | Content |
|---|------|---------|
| 1 | `system` | Instructions: who the model is, what tools exist, when to call them |
| 2 | `user` | The user's question |
| 3 | `assistant` (with `tool_calls`) | Model decides to call a tool, emits structured arguments |
| 4 | `tool` | Your code runs the tool and feeds the result back as a message |
| 5 | `assistant` | Final natural-language answer |

Steps 1–3 are one `ollama.chat()` call. Steps 4–5 are a second `ollama.chat()` call with the conversation extended.

Below is a system prompt that's deliberately short. Long system prompts hurt small models more than they help.

In [3]:
SYSTEM_PROMPT = (
    "You are a CTI analyst assistant with access to a small set of tools that "
    "query a CTI corpus (DuckDB) and a public ransomware feed. "
    "Rules:\n"
    "- If a tool can answer the question, call exactly one tool.\n"
    "- If no tool is relevant, answer directly from general knowledge.\n"
    "- Keep answers short and factual. Cite url_id or alert_id when relevant."
)
print(SYSTEM_PROMPT)

You are a CTI analyst assistant with access to a small set of tools that query a CTI corpus (DuckDB) and a public ransomware feed. Rules:
- If a tool can answer the question, call exactly one tool.
- If no tool is relevant, answer directly from general knowledge.
- Keep answers short and factual. Cite url_id or alert_id when relevant.


## 3. One full exchange, with full visibility

We make the model's behaviour fully visible: print the assistant message (with any tool calls), the tool result, and the final answer. This is the part most agentic-AI tutorials hide behind a wrapper. Don't hide it yet.

In [4]:
def ask_one_step(question, *, model=MODEL, verbose=True):
    """Single-step tool use: model picks at most one tool, then answers.

    Returns the final assistant message string. If verbose, prints each
    intermediate piece so you can see what the model decided.
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    # Round 1: model sees tools, decides whether to call one.
    resp = ollama.chat(
        model=model,
        messages=messages,
        tools=TOOL_SCHEMAS,
        options={"temperature": 0.2, "num_ctx": 4096},
    )
    msg = resp["message"]
    messages.append(msg)

    tool_calls = msg.get("tool_calls") or []
    if verbose:
        print("--- assistant (round 1) ---")
        if tool_calls:
            for tc in tool_calls:
                fn = tc["function"]
                print(f"  tool_call: {fn['name']}({fn.get('arguments', {})})")
        else:
            print("  (no tool call)")
            print("  content:", (msg.get("content") or "")[:300])

    # If the model did not call a tool, we're done.
    if not tool_calls:
        return msg.get("content") or ""

    # Single-step: take only the first tool call (ignore extras).
    tc = tool_calls[0]
    fn_name = tc["function"]["name"]
    fn_args = tc["function"].get("arguments") or {}
    if isinstance(fn_args, str):
        try:
            fn_args = json.loads(fn_args)
        except json.JSONDecodeError:
            fn_args = {}

    tool_result = dispatch_tool_call(fn_name, fn_args)
    if verbose:
        print("--- tool result ---")
        print("  ", json.dumps(tool_result)[:400])

    messages.append({
        "role": "tool",
        "name": fn_name,
        "content": json.dumps(tool_result),
    })

    # Round 2: model reads the tool result and produces a final answer.
    resp2 = ollama.chat(
        model=model,
        messages=messages,
        options={"temperature": 0.2, "num_ctx": 4096},
    )
    final = resp2["message"].get("content") or ""
    if verbose:
        print("--- assistant (round 2, final) ---")
        print(" ", final[:600])
    return final

## 4. Run a few questions

Five questions, each designed to probe a different tool or behaviour:

1. **Triggers `query_documents`** with a label filter
2. **Triggers `query_alerts`** with severity filter
3. **Triggers `count_recent_victims`** (network tool)
4. **Triggers `get_document`** — model has to chain in spirit, but here we hand-feed a known url_id
5. **Should NOT trigger any tool** — generic question, model should answer from general knowledge

Local 8B models will get some of these wrong on first run. That's expected. nb 06 is dedicated to characterising those failures.

In [5]:
_ = ask_one_step("How many documents in the corpus are labeled 'Hacking'?")

--- assistant (round 1) ---
  tool_call: query_documents({'label': 'Hacking'})
--- tool result ---
   {"count": 1, "results": [{"url_id": "f77e59ad053e076d99eecfabd4f2ed043c5c8807", "source": "onion", "label": "Hacking", "score": 0.964, "title": "Share and accept documents securely"}]}
--- assistant (round 2, final) ---
  The query returned 1 document labeled 'Hacking'.


In [6]:
_ = ask_one_step("What high-severity alerts have been raised in the last 30 days?")

--- assistant (round 1) ---
  tool_call: query_alerts({'limit': '10', 'severity': 'high', 'since_days': '30'})
--- tool result ---
   {"count": 1, "results": [{"alert_id": "f77e59ad053e076d99eecfabd4f2ed043c5c8807", "severity": "high", "source": "onion", "label": "Hacking", "score": 0.964, "title": "Share and accept documents securely", "raised_at": "2026-04-29 20:43:38.529845"}]}
--- assistant (round 2, final) ---
  Based on the query results, there is one high-severity alert raised in the last 30 days with an alert_id of "f77e59ad053e076d99eecfabd4f2ed043c5c8807". The alert was raised by the source "onion" and has a label of "Hacking" with a score of 0.964.


In [7]:
_ = ask_one_step("How many ransomware victims have been posted in the last 14 days?")

--- assistant (round 1) ---
  tool_call: count_recent_victims({'lookback_days': '14'})
--- tool result ---
   {"total_returned_by_api": 100, "in_lookback_window": 0, "lookback_days": 14}


KeyboardInterrupt: 

In [ ]:
# Pick a real url_id from the corpus, then ask about it directly.
first_uid = query_documents(limit=1)["results"][0]["url_id"]
print("asking about url_id:", first_uid)
_ = ask_one_step(f"Get document {first_uid} and summarize it in one sentence.")

In [ ]:
# Should NOT call any tool.
_ = ask_one_step("What does the acronym TTP stand for in CTI?")

## 5. Things to watch for

After you run the cells above, look for these patterns. They are the bread-and-butter failure modes of local 8B tool-use:

- **Wrong tool chosen.** Model calls `query_documents` when `query_alerts` was the right answer. Usually fixable with a sharper system prompt or tool description.
- **Bad arguments.** Model passes `severity="critical"` when the schema says `enum: ["high", "medium"]`. Our dispatcher catches this and returns an error — but a stronger model would have read the schema better.
- **Unnecessary tool call.** Model calls a tool for question 5 instead of answering directly.
- **Hallucinated answer despite tool result.** Model gets clean tool output but ignores it. Rare on Llama 3.1, common on smaller models.

Each of these is a real teaching moment — note them down. nb 06 will replay them and propose mitigations.

## What's next

nb 03 turns this single-step exchange into a **planning loop**. The model can now keep calling tools until it decides it has enough information, or until a hard step-cap stops it. That cap is the most important line in nb 03 — without it, an 8B model will sometimes loop forever on a confusing question.